In [9]:
import sys
import os

sys.path.append(os.path.abspath(".."))

from config.spark import spark


In [10]:
from pyspark.sql.types import StructType,StructField,StringType,IntegerType

customer_schema = StructType([
    StructField("customer_id",StringType()),
    StructField("customer_unique_id",StringType()),
    StructField("customer_zip_code_prefix",IntegerType()),
    StructField("customer_city",StringType()),
    StructField("customer_state",StringType())
])


In [11]:
customers_df = (
    spark.read.format('csv')
    .option('header','True')
   # .option('inferschema','True')
    .schema(customer_schema)
   
    .load('../landing/files/olist_customers_dataset.csv')
    
)

In [12]:
from pyspark.sql.functions import current_timestamp

customers_df = customers_df.withColumn("ingestion_time",current_timestamp())

In [13]:
customers_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- ingestion_time: timestamp (nullable = false)



In [14]:
customers_df.write.mode("overwrite").saveAsTable("bronze.customers")

In [16]:
spark.sql(
    """select * from bronze.customers
    """
).show()

+--------------------+--------------------+------------------------+--------------------+--------------+--------------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|       customer_city|customer_state|      ingestion_time|
+--------------------+--------------------+------------------------+--------------------+--------------+--------------------+
|f2a1d75b74d9ec748...|15ee900ec703c9a10...|                   68590|             jacunda|            PA|2026-10-17 14:42:...|
|f15272fe9d0e2ae32...|11e74a9cbe1158d1c...|                   15056|sao jose do rio p...|            SP|2026-10-17 14:42:...|
|7324ecb0ff143f561...|c6be127fa6e30c6f7...|                   13302|                 itu|            SP|2026-10-17 14:42:...|
|7accf3d920f47c07f...|a7f1a6dc9ba06844b...|                   45638|             coaraci|            BA|2026-10-17 14:42:...|
|3680a273ddb333253...|6cbfcc29787035834...|                   29700|            colatina|            ES|2026-10-17 14: